In [0]:
-- Genie Active Users + T7D $DBU – by AE Email – Daily Trend
WITH ae_accounts AS (
  SELECT DISTINCT deployable_account_name
  FROM main.gtm_gold.account_consumption_daily
  WHERE horizontal_and_vertical_hierarchy_concatenated_emails LIKE CONCAT('%', :ae_email, '%')
    AND YEAR(usage_date) + CASE WHEN MONTH(usage_date) >= 2 THEN 1 ELSE 0 END >= 2026
),
genie_kpis AS (
  SELECT
    g.date,
    h.deployable_account_name,
    SUM(g.T7D_Users)  AS genie_t7d_users,
    SUM(g.T28D_Users) AS genie_t28d_users
  FROM main.eng_datarooms.genie_daily_kpis g
  INNER JOIN main.fin_live_gold.sfdc_hierarchy_mapping h
    ON g.salesforce_account_name = h.sfdc_account_name
  INNER JOIN ae_accounts a
    ON h.deployable_account_name = a.deployable_account_name
  WHERE YEAR(g.date) + CASE WHEN MONTH(g.date) >= 2 THEN 1 ELSE 0 END >= 2026
  GROUP BY g.date, h.deployable_account_name
),
genie_dbu AS (
  SELECT
    usage_date,
    deployable_account_name,
    SUM(genie_standalone_dbu_dollars)               AS genie_dbu_dollars,
    SUM(genie_standalone_dbu_dollars_t7d_sum)       AS genie_t7d_dbu_dollars,
    SUM(genie_standalone_dbu_dollars_t7d_sum_prev)  AS genie_t7d_dbu_dollars_prev,
    SUM(genie_standalone_dbu_dollars_t28d_sum)      AS genie_t28d_dbu_dollars,
    SUM(genie_standalone_dbu_dollars_t28d_sum_prev) AS genie_t28d_dbu_dollars_prev
  FROM main.gtm_gold.account_consumption_daily
  WHERE horizontal_and_vertical_hierarchy_concatenated_emails LIKE CONCAT('%', :ae_email, '%')
  AND Business_Unit = :business_unit
  AND sales_subregion_level_1 = :region_level_1
  AND sales_subregion_level_2 = :region_level_2  
  AND YEAR(usage_date) + CASE WHEN MONTH(usage_date) >= 2 THEN 1 ELSE 0 END >= 2026
  GROUP BY usage_date, deployable_account_name
),
fy_start AS (
  -- First date available in genie_daily_kpis >= start of the FY containing the latest snapshot
  --calculates the first day of the current fiscal.
  SELECT MIN(date) AS fy_first_date
  FROM main.eng_datarooms.genie_daily_kpis
  WHERE date >= MAKE_DATE(
    CASE WHEN MONTH((SELECT MAX(date) FROM main.eng_datarooms.genie_daily_kpis)) >= 2
         THEN YEAR((SELECT MAX(date) FROM main.eng_datarooms.genie_daily_kpis))
         ELSE YEAR((SELECT MAX(date) FROM main.eng_datarooms.genie_daily_kpis)) - 1
    END, 2, 1)
),
fy_start_kpis AS (
  -- T28D users per account on the first day of the current FY
  SELECT
    h.deployable_account_name,
    SUM(g.T28D_Users) AS fy_start_t28d_users
  FROM main.eng_datarooms.genie_daily_kpis g
  INNER JOIN main.fin_live_gold.sfdc_hierarchy_mapping h
    ON g.salesforce_account_name = h.sfdc_account_name
  INNER JOIN ae_accounts a
    ON h.deployable_account_name = a.deployable_account_name
  CROSS JOIN fy_start fs
  WHERE g.date = fs.fy_first_date
  GROUP BY h.deployable_account_name
)

SELECT
  k.date,
  YEAR(k.date) + CASE WHEN MONTH(k.date) >= 2 THEN 1 ELSE 0 END AS fiscal_year,
  CONCAT(
    'FY', RIGHT(CAST(YEAR(k.date) + CASE WHEN MONTH(k.date) >= 2 THEN 1 ELSE 0 END AS STRING), 2),
    '-',
    CASE
      WHEN MONTH(k.date) IN (2,3,4)   THEN 'Q1'
      WHEN MONTH(k.date) IN (5,6,7)   THEN 'Q2'
      WHEN MONTH(k.date) IN (8,9,10)  THEN 'Q3'
      ELSE                                 'Q4'
    END
  )  AS fiscal_quarter,
  CASE WHEN k.date = (SELECT MAX(date) FROM genie_kpis) THEN 'Y' ELSE 'N' END AS latest_snapshot,
  k.deployable_account_name,
  k.genie_t7d_users,
  k.genie_t28d_users,
  k.genie_t28d_users - f.fy_start_t28d_users as t28d_users_ytd_diff,
  fy_start_t28d_users,
  --ROUND(100.0 * (k.genie_t28d_users - f.fy_start_t28d_users) / NULLIF(f.fy_start_t28d_users, 0), 1) AS t28d_users_growth_pct,
  d.genie_dbu_dollars,
  d.genie_t7d_dbu_dollars,
  d.genie_t7d_dbu_dollars_prev,
  d.genie_t28d_dbu_dollars,
  d.genie_t28d_dbu_dollars_prev
FROM genie_kpis k
LEFT JOIN genie_dbu d
  ON  k.deployable_account_name = d.deployable_account_name
  AND k.date = d.usage_date
LEFT JOIN fy_start_kpis f
  ON  k.deployable_account_name = f.deployable_account_name
WHERE (DAYOFWEEK(k.date) = 6                          -- Fridays only
   OR k.date = (SELECT MAX(date) FROM genie_kpis))    -- plus latest available day
ORDER BY k.date, k.deployable_account_name

Databricks visualization. Run in Databricks to view.